In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', 'lightgbm', '-q'])

import os, time, warnings
from datetime import datetime
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from scipy import stats

from sklearn.linear_model    import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.preprocessing   import RobustScaler
from sklearn.pipeline        import make_pipeline, Pipeline
from sklearn.metrics         import mean_squared_error, r2_score
import joblib

import xgboost  as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
sns.set_style('whitegrid')
print('libraries loaded')


In [ ]:
CFG = dict(
    train_path    = '/content/drive/MyDrive/house_prices/train.csv',
    test_path     = '/content/drive/MyDrive/house_prices/test.csv',
    output_dir    = '/content',
    seed          = 42,
    n_folds       = 5,
    outlier_sf    = 4000,
    outlier_price = 300_000,
    skew_thresh   = 0.75,
    ridge_alpha   = 10,
    lasso_alpha   = 0.0005,
    enet_alpha    = 0.0005,
    enet_l1       = 0.5,
    rf_trees      = 400,
    rf_depth      = 15,
    rf_min_split  = 5,
    rf_min_leaf   = 2,
    gbr_trees     = 3000,
    gbr_lr        = 0.05,
    gbr_depth     = 4,
    gbr_subsample = 0.8,
    xgb_trees     = 3000,
    xgb_lr        = 0.05,
    xgb_depth     = 4,
    xgb_alpha     = 0.5,
    xgb_lambda    = 0.8,
    xgb_colsample = 0.4,
    xgb_subsample = 0.8,
    lgb_trees     = 3000,
    lgb_lr        = 0.05,
    lgb_leaves    = 31,
    lgb_alpha     = 0.5,
    lgb_lambda    = 0.8,
    lgb_colsample = 0.4,
    lgb_subsample = 0.8,
    early_stop    = 50,
)

np.random.seed(CFG['seed'])
import random; random.seed(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(RUN_TIMESTAMP)


In [ ]:
train = pd.read_csv(CFG['train_path'])
test  = pd.read_csv(CFG['test_path'])

assert 'SalePrice' in train.columns,       'train.csv missing SalePrice'
assert 'SalePrice' not in test.columns,    'test.csv should not have SalePrice'
assert 'Id' in train.columns,              'Id missing from train'
assert 'Id' in test.columns,               'Id missing from test'
assert train['Id'].nunique() == len(train), 'duplicate Ids in train'
assert test['Id'].nunique()  == len(test),  'duplicate Ids in test'

test_ids = test['Id'].values

print(f'train : {train.shape[0]} rows, {train.shape[1]} cols')
print(f'test  : {test.shape[0]} rows, {test.shape[1]} cols')
train.head(3)


In [ ]:
print(train.dtypes.value_counts().to_string())
print()
print(train['SalePrice'].describe().apply(lambda x: f'${x:,.0f}' if x > 1 else f'{x:.4f}'))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(train['SalePrice'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice — original')
axes[0].set_xlabel('SalePrice ($)')

axes[1].hist(np.log1p(train['SalePrice']), bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('log(SalePrice+1)')
axes[1].set_xlabel('log(SalePrice + 1)')

log_price = np.log1p(train['SalePrice'])
(osm, osr), (slope, intercept, r) = stats.probplot(log_price, dist='norm')
axes[2].scatter(osm, osr, alpha=0.4, s=10, color='mediumseagreen')
axes[2].plot(osm, slope*np.array(osm)+intercept, color='red', linewidth=1.5)
axes[2].set_title(f'QQ plot — log(SalePrice)  R2={r**2:.4f}')
axes[2].set_xlabel('Theoretical quantiles')
axes[2].set_ylabel('Sample quantiles')

plt.tight_layout(); plt.show()
print(f'skew original  : {train["SalePrice"].skew():.4f}')
print(f'skew log       : {log_price.skew():.4f}')


In [ ]:
miss_train = train.isnull().sum()
miss_test  = test.isnull().sum()
miss_train = miss_train[miss_train > 0].sort_values(ascending=False)
miss_pct   = (miss_train / len(train) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
miss_pct.head(20).sort_values().plot(kind='barh', color='salmon', edgecolor='white', ax=axes[0])
axes[0].set_title('Missing values — train')
axes[0].set_xlabel('Missing %')

miss_test_pct = (miss_test[miss_test > 0] / len(test) * 100).sort_values(ascending=False).head(20)
miss_test_pct.sort_values().plot(kind='barh', color='lightcoral', edgecolor='white', ax=axes[1])
axes[1].set_title('Missing values — test')
axes[1].set_xlabel('Missing %')

plt.tight_layout(); plt.show()
print(f'train cols with missing : {len(miss_train)}')
print(f'test  cols with missing : {len(miss_test[miss_test>0])}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(x='OverallQual', y='SalePrice', data=train, palette='viridis', ax=axes[0,0])
axes[0,0].set_title('Overall quality vs sale price')
axes[0,0].set_xlabel('Quality (1=very poor, 10=very excellent)')

outs = train[(train['GrLivArea'] > CFG['outlier_sf']) & (train['SalePrice'] < CFG['outlier_price'])]
axes[0,1].scatter(train['GrLivArea'], train['SalePrice'], alpha=0.35, c='steelblue', s=12)
axes[0,1].scatter(outs['GrLivArea'], outs['SalePrice'], c='red', s=40, zorder=5, label='outliers')
axes[0,1].legend()
axes[0,1].set_title('GrLivArea vs sale price')
axes[0,1].set_xlabel('Above-ground living area (sq ft)')

axes[1,0].scatter(train['YearBuilt'], train['SalePrice'], alpha=0.25, c='mediumseagreen', s=12)
axes[1,0].set_title('Year built vs sale price')
axes[1,0].set_xlabel('Year built')

axes[1,1].scatter(train['TotalBsmtSF'], train['SalePrice'], alpha=0.3, c='coral', s=12)
axes[1,1].set_title('Total basement SF vs sale price')
axes[1,1].set_xlabel('Total basement SF')

for ax in axes.flat: ax.set_ylabel('Sale price ($)')
plt.tight_layout(); plt.show()


In [ ]:
num_cols  = train.select_dtypes(include=[np.number]).columns
corr      = train[num_cols].corr()
top_feats = corr['SalePrice'].abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(13, 11))
sns.heatmap(train[top_feats].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation — top 15 numeric features')
plt.tight_layout(); plt.show()


In [ ]:
neigh_map = {
    'Blmngtn':'Bloomington Hts','Blueste':'Bluestem','BrDale':'Briardale',
    'BrkSide':'Brookside','ClearCr':'Clear Creek','CollgCr':'College Creek',
    'Crawfor':'Crawford','Edwards':'Edwards','Gilbert':'Gilbert',
    'IDOTRR':'Iowa DOT/RR','MeadowV':'Meadow Village','Mitchel':'Mitchell',
    'Names':'North Ames','NoRidge':'Northridge','NPkVill':'Northpark Villa',
    'NridgHt':'Northridge Hts','NWAmes':'NW Ames','OldTown':'Old Town',
    'SWISU':'SW ISU','Sawyer':'Sawyer','SawyerW':'Sawyer West',
    'Somerst':'Somerset','StoneBr':'Stone Brook','Timber':'Timberland',
    'Veenker':'Veenker'
}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

neigh_stats = (train.groupby('Neighborhood')['SalePrice']
               .agg(['median','std'])
               .rename(neigh_map)
               .sort_values('median', ascending=False))
neigh_stats['median'].plot(kind='bar', color='coral', edgecolor='white',
                           yerr=neigh_stats['std'], capsize=3, ax=axes[0])
axes[0].set_title('Median sale price by neighborhood')
axes[0].set_ylabel('Sale price ($)')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)

sns.boxplot(x='BldgType', y='SalePrice', data=train, palette='Set2', ax=axes[1])
axes[1].set_title('Building type vs sale price')
axes[1].set_xlabel('1Fam=single | 2FmCon=2-family | Duplx=duplex | TwnhsE/I=townhouse')
axes[1].set_ylabel('Sale price ($)')

plt.tight_layout(); plt.show()


In [ ]:
outlier_mask = ~(
    (train['GrLivArea'] > CFG['outlier_sf']) &
    (train['SalePrice'] < CFG['outlier_price'])
)
train_clean = train[outlier_mask].reset_index(drop=True)
y = np.log1p(train_clean['SalePrice'])

print(f'outliers removed : {(~outlier_mask).sum()}')
print(f'training samples : {len(train_clean)}')
print(f'y skew           : {y.skew():.4f}')

train_feats = train_clean.drop(['Id', 'SalePrice'], axis=1).copy()
test_feats  = test.drop('Id', axis=1).copy()

assert len(train_feats) == len(y)
print(f'train_feats : {train_feats.shape}')
print(f'test_feats  : {test_feats.shape}')


In [ ]:
def impute(train_df, test_df):
    tr, te = train_df.copy(), test_df.copy()

    cat_none = ['PoolQC','Fence','MiscFeature','Alley','FireplaceQu',
                'GarageType','GarageFinish','GarageQual','GarageCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                'MasVnrType']
    for c in cat_none:
        if c in tr.columns: tr[c] = tr[c].fillna('None')
        if c in te.columns: te[c] = te[c].fillna('None')

    num_zero = ['GarageYrBlt','GarageArea','GarageCars',
                'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF',
                'BsmtFullBath','BsmtHalfBath','MasVnrArea','PoolArea']
    for c in num_zero:
        if c in tr.columns: tr[c] = tr[c].fillna(0)
        if c in te.columns: te[c] = te[c].fillna(0)

    lot_med = tr.groupby('Neighborhood')['LotFrontage'].median()
    tr['LotFrontage'] = tr['LotFrontage'].fillna(tr['Neighborhood'].map(lot_med))
    te['LotFrontage'] = te['LotFrontage'].fillna(te['Neighborhood'].map(lot_med))
    global_lot_med = tr['LotFrontage'].median()
    tr['LotFrontage'] = tr['LotFrontage'].fillna(global_lot_med)
    te['LotFrontage'] = te['LotFrontage'].fillna(global_lot_med)

    elec_mode = tr['Electrical'].mode()[0]
    tr['Electrical'] = tr['Electrical'].fillna(elec_mode)
    te['Electrical'] = te['Electrical'].fillna(elec_mode)

    tr.drop(columns=['Utilities'], inplace=True, errors='ignore')
    te.drop(columns=['Utilities'], inplace=True, errors='ignore')

    cat_modes   = {c: tr[c].mode()[0] for c in tr.select_dtypes('object').columns}
    num_medians = {c: tr[c].median()  for c in tr.select_dtypes(np.number).columns}

    for c, v in cat_modes.items():
        tr[c] = tr[c].fillna(v)
        if c in te.columns: te[c] = te[c].fillna(v)
    for c, v in num_medians.items():
        tr[c] = tr[c].fillna(v)
        if c in te.columns: te[c] = te[c].fillna(v)

    return tr, te

train_feats, test_feats = impute(train_feats, test_feats)
print(f'train NaN : {train_feats.isnull().sum().sum()}')
print(f'test  NaN : {test_feats.isnull().sum().sum()}')


In [ ]:
def engineer_features(df):
    df = df.copy()
    df['HouseAge']     = df['YrSold'] - df['YearBuilt']
    df['RemodelAge']   = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodelled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    df['IsNew']        = (df['YrSold'] == df['YearBuilt']).astype(int)
    df['TotalBath']    = (df['FullBath'] + 0.5*df['HalfBath'] +
                          df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalPorch']   = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                          df['3SsnPorch']   + df['ScreenPorch'])
    df['HasPool']      = (df['PoolArea']    > 0).astype(int)
    df['Has2ndFloor']  = (df['2ndFlrSF']    > 0).astype(int)
    df['HasGarage']    = (df['GarageArea']  > 0).astype(int)
    df['HasBsmt']      = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces']  > 0).astype(int)
    df['HasPorch']     = (df['TotalPorch']  > 0).astype(int)
    df['GarageAge']    = (df['YrSold'] - df['GarageYrBlt']).clip(lower=0)
    df['LivAreaRatio'] = df['GrLivArea'] / (df['TotalSF'] + 1)
    df['QualSF']       = df['OverallQual'] * df['TotalSF']
    df['QualAge']      = df['OverallQual'] * df['HouseAge']
    return df

train_feats = engineer_features(train_feats)
test_feats  = engineer_features(test_feats)
print(f'features : {train_feats.shape[1]}')


In [ ]:
def ordinal_encode(df):
    df = df.copy()
    qual5 = {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5}
    for c in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
              'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']:
        if c in df.columns:
            df[c] = df[c].map(qual5).fillna(0).astype(int)
    df['LotShape']     = df['LotShape'].map({'Reg':0,'IR1':1,'IR2':2,'IR3':3}).fillna(0)
    df['LandSlope']    = df['LandSlope'].map({'Gtl':0,'Mod':1,'Sev':2}).fillna(0)
    df['LandContour']  = df['LandContour'].map({'Low':0,'Bnk':1,'HLS':2,'Lvl':3}).fillna(3)
    df['BsmtExposure'] = df['BsmtExposure'].map({'None':0,'No':0,'Mn':1,'Av':2,'Gd':3}).fillna(0)
    bsmt_fin = {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6}
    df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_fin).fillna(0)
    df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_fin).fillna(0)
    df['GarageFinish'] = df['GarageFinish'].map({'None':0,'Unf':1,'RFn':2,'Fin':3}).fillna(0)
    df['Fence']        = df['Fence'].map({'None':0,'MnWw':1,'GdWo':2,'MnPrv':3,'GdPrv':4}).fillna(0)
    df['PavedDrive']   = df['PavedDrive'].map({'N':0,'P':1,'Y':2}).fillna(0)
    df['CentralAir']   = df['CentralAir'].map({'N':0,'Y':1})
    df['Functional']   = df['Functional'].map({'Sal':0,'Sev':1,'Maj2':2,'Maj1':3,
                                               'Mod':4,'Min2':5,'Min1':6,'Typ':7}).fillna(7)
    return df

train_feats = ordinal_encode(train_feats)
test_feats  = ordinal_encode(test_feats)
print('ordinal encoding done')


In [ ]:
train_enc = pd.get_dummies(train_feats)
test_enc  = pd.get_dummies(test_feats)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

assert list(train_enc.columns) == list(test_enc.columns)
print(f'encoded features : {train_enc.shape[1]}')
print(f'train_enc : {train_enc.shape}')
print(f'test_enc  : {test_enc.shape}')


In [ ]:
num_cols    = train_enc.select_dtypes(include=[np.number]).columns.tolist()
train_skews = train_enc[num_cols].apply(lambda x: skew(x.dropna()))
skewed_cols = train_skews[train_skews.abs() > CFG['skew_thresh']].index.tolist()

train_enc[skewed_cols] = np.log1p(train_enc[skewed_cols].clip(lower=0))
test_enc[skewed_cols]  = np.log1p(test_enc[skewed_cols].clip(lower=0))
print(f'skew-corrected features : {len(skewed_cols)}')


In [ ]:
X_train = train_enc.reset_index(drop=True)
X_test  = test_enc.reset_index(drop=True)
y       = y.reset_index(drop=True)

assert len(X_train) == len(y)
assert X_train.shape[1] == X_test.shape[1]
assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum()  == 0

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y       : {len(y)}')


In [ ]:
kf = KFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

results       = {}
fitted_models = {}

def rmse_cv(model, X, y, cv=kf):
    scores = cross_val_score(model, X, y,
                             scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
    rmse = np.sqrt(-scores)
    return rmse.mean(), rmse.std()

def report(name, model, X, y, m, s):
    t0    = time.time()
    model.fit(X, y)
    preds      = model.predict(X)
    train_rmse = np.sqrt(mean_squared_error(y, preds))
    train_r2   = r2_score(y, preds)
    fitted_models[name] = model
    print(f"{name:<22}  cv: {m:.4f} +/-{s:.4f}  train rmse: {train_rmse:.4f}  r2: {train_r2:.4f}  {time.time()-t0:.0f}s")
    return model

print(f"{'model':<22}  {'cv rmse':>10}  {'train rmse':>12}  {'r2':>6}  time")
print('-' * 70)


In [ ]:
lr = make_pipeline(RobustScaler(), LinearRegression(fit_intercept=True, n_jobs=-1))
m, s = rmse_cv(lr, X_train, y)
results['Linear Regression'] = m
report('Linear Regression', lr, X_train, y, m, s)


In [ ]:
ridge = make_pipeline(
    RobustScaler(),
    Ridge(alpha=CFG['ridge_alpha'], fit_intercept=True, random_state=CFG['seed'])
)
m, s = rmse_cv(ridge, X_train, y)
results['Ridge'] = m
report('Ridge', ridge, X_train, y, m, s)


In [ ]:
lasso = make_pipeline(
    RobustScaler(),
    Lasso(alpha=CFG['lasso_alpha'], max_iter=10000, tol=1e-4,
          selection='random', random_state=CFG['seed'])
)
m, s = rmse_cv(lasso, X_train, y)
results['Lasso'] = m
report('Lasso', lasso, X_train, y, m, s)
n_nz = np.sum(lasso.named_steps['lasso'].coef_ != 0)
print(f'  lasso selected {n_nz} / {X_train.shape[1]} features')


In [ ]:
enet = make_pipeline(
    RobustScaler(),
    ElasticNet(alpha=CFG['enet_alpha'], l1_ratio=CFG['enet_l1'],
               max_iter=10000, tol=1e-4, selection='random',
               random_state=CFG['seed'])
)
m, s = rmse_cv(enet, X_train, y)
results['ElasticNet'] = m
report('ElasticNet', enet, X_train, y, m, s)


In [ ]:
rf = RandomForestRegressor(
    n_estimators=CFG['rf_trees'],
    max_depth=CFG['rf_depth'],
    min_samples_split=CFG['rf_min_split'],
    min_samples_leaf=CFG['rf_min_leaf'],
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=CFG['seed'],
    n_jobs=-1,
)
m, s = rmse_cv(rf, X_train, y)
results['Random Forest'] = m
report('Random Forest', rf, X_train, y, m, s)
print(f'  oob r2 : {rf.oob_score_:.4f}')


In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=CFG['gbr_trees'],
    learning_rate=CFG['gbr_lr'],
    max_depth=CFG['gbr_depth'],
    subsample=CFG['gbr_subsample'],
    max_features='sqrt',
    min_samples_split=5,
    min_samples_leaf=2,
    validation_fraction=0.1,
    n_iter_no_change=60,
    tol=1e-4,
    random_state=CFG['seed'],
)
m, s = rmse_cv(gbr, X_train, y)
results['Gradient Boosting'] = m
report('Gradient Boosting', gbr, X_train, y, m, s)
print(f'  trees used : {gbr.n_estimators_}')


In [ ]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=CFG['xgb_trees'],
    learning_rate=CFG['xgb_lr'],
    max_depth=CFG['xgb_depth'],
    min_child_weight=2,
    subsample=CFG['xgb_subsample'],
    colsample_bytree=CFG['xgb_colsample'],
    gamma=0.1,
    reg_alpha=CFG['xgb_alpha'],
    reg_lambda=CFG['xgb_lambda'],
    tree_method='hist',
    eval_metric='rmse',
    early_stopping_rounds=CFG['early_stop'],
    n_jobs=-1,
    random_state=CFG['seed'],
    verbosity=0,
)

xgb_scores, best_iters = [], []
t0 = time.time()
for tr_idx, val_idx in kf.split(X_train):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y.iloc[tr_idx],       y.iloc[val_idx]
    xgb_model.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
    xgb_scores.append(np.sqrt(mean_squared_error(yval, xgb_model.predict(Xval))))
    best_iters.append(xgb_model.best_iteration)

m, s   = np.mean(xgb_scores), np.std(xgb_scores)
best_n = int(np.mean(best_iters) * 1.05)

xgb_final = xgb.XGBRegressor(
    objective='reg:squarederror', n_estimators=best_n,
    learning_rate=CFG['xgb_lr'], max_depth=CFG['xgb_depth'],
    min_child_weight=2, subsample=CFG['xgb_subsample'],
    colsample_bytree=CFG['xgb_colsample'], gamma=0.1,
    reg_alpha=CFG['xgb_alpha'], reg_lambda=CFG['xgb_lambda'],
    tree_method='hist', n_jobs=-1, random_state=CFG['seed'], verbosity=0,
)
xgb_final.fit(X_train, y)
fitted_models['XGBoost'] = xgb_final
results['XGBoost'] = m

preds      = xgb_final.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y, preds))
train_r2   = r2_score(y, preds)
print(f"{'XGBoost':<22}  cv: {m:.4f} +/-{s:.4f}  train rmse: {train_rmse:.4f}  r2: {train_r2:.4f}  {time.time()-t0:.0f}s")
print(f'  best iteration (mean across folds) : {int(np.mean(best_iters))}  ->  refit with {best_n} trees')


In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=CFG['lgb_trees'],
    learning_rate=CFG['lgb_lr'],
    num_leaves=CFG['lgb_leaves'],
    min_child_samples=20,
    colsample_bytree=CFG['lgb_colsample'],
    subsample=CFG['lgb_subsample'],
    subsample_freq=1,
    reg_alpha=CFG['lgb_alpha'],
    reg_lambda=CFG['lgb_lambda'],
    random_state=CFG['seed'],
    n_jobs=-1,
    verbose=-1,
)

lgb_scores, lgb_best_iters = [], []
t0 = time.time()
for tr_idx, val_idx in kf.split(X_train):
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y.iloc[tr_idx],       y.iloc[val_idx]
    lgb_model.fit(
        Xtr, ytr,
        eval_set=[(Xval, yval)],
        callbacks=[lgb.early_stopping(CFG['early_stop'], verbose=False),
                   lgb.log_evaluation(-1)],
    )
    lgb_scores.append(np.sqrt(mean_squared_error(yval, lgb_model.predict(Xval))))
    lgb_best_iters.append(lgb_model.best_iteration_)

m, s      = np.mean(lgb_scores), np.std(lgb_scores)
lgb_best_n = int(np.mean(lgb_best_iters) * 1.05)

lgb_final = lgb.LGBMRegressor(
    n_estimators=lgb_best_n,
    learning_rate=CFG['lgb_lr'],
    num_leaves=CFG['lgb_leaves'],
    min_child_samples=20,
    colsample_bytree=CFG['lgb_colsample'],
    subsample=CFG['lgb_subsample'],
    subsample_freq=1,
    reg_alpha=CFG['lgb_alpha'],
    reg_lambda=CFG['lgb_lambda'],
    random_state=CFG['seed'],
    n_jobs=-1,
    verbose=-1,
)
lgb_final.fit(X_train, y)
fitted_models['LightGBM'] = lgb_final
results['LightGBM'] = m

preds      = lgb_final.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y, preds))
train_r2   = r2_score(y, preds)
print(f"{'LightGBM':<22}  cv: {m:.4f} +/-{s:.4f}  train rmse: {train_rmse:.4f}  r2: {train_r2:.4f}  {time.time()-t0:.0f}s")
print(f'  best iteration (mean across folds) : {int(np.mean(lgb_best_iters))}  ->  refit with {lgb_best_n} trees')


In [ ]:
results_df = (pd.DataFrame.from_dict(results, orient='index', columns=['CV RMSE'])
               .sort_values('CV RMSE'))
results_df['approx % error'] = results_df['CV RMSE'].apply(
    lambda r: f'{(np.expm1(r))*100:.1f}%'
)

for rank, (name, row) in enumerate(results_df.iterrows(), 1):
    print(f"{rank}. {name:<22} {row['CV RMSE']:.4f}   {row['approx % error']}")

sorted_names = results_df['CV RMSE'].sort_values(ascending=True).index.tolist()
palette = ['gold' if i == 0 else 'silver' if i == 1 else 'peru' if i == 2 else 'steelblue'
           for i in range(len(sorted_names))]

fig, ax = plt.subplots(figsize=(11, 5))
results_df['CV RMSE'].sort_values(ascending=True).plot(
    kind='barh', color=palette, edgecolor='white', ax=ax)
ax.set_title('CV RMSE on log(SalePrice) — lower is better')
ax.set_xlabel('CV RMSE')
for bar in ax.patches:
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.4f}", va='center', fontsize=9)
plt.tight_layout(); plt.show()

best_name = results_df.index[0]
print(f'best model : {best_name}  ({results_df.iloc[0,0]:.4f})')


In [ ]:
best_model = fitted_models[best_name]

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y,
    cv=KFold(n_splits=5, shuffle=True, random_state=CFG['seed']),
    scoring='neg_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=1,
)

tr_rmse  = np.sqrt(-train_scores)
val_rmse = np.sqrt(-val_scores)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, tr_rmse.mean(axis=1),  'o-', color='steelblue',  label='train')
plt.fill_between(train_sizes,
                 tr_rmse.mean(axis=1) - tr_rmse.std(axis=1),
                 tr_rmse.mean(axis=1) + tr_rmse.std(axis=1), alpha=0.15, color='steelblue')
plt.plot(train_sizes, val_rmse.mean(axis=1), 'o-', color='darkorange', label='cv')
plt.fill_between(train_sizes,
                 val_rmse.mean(axis=1) - val_rmse.std(axis=1),
                 val_rmse.mean(axis=1) + val_rmse.std(axis=1), alpha=0.15, color='darkorange')
gap  = val_rmse.mean(axis=1)[-1] - tr_rmse.mean(axis=1)[-1]
diag = 'overfitting' if gap > 0.03 else 'underfitting' if val_rmse.mean(axis=1)[-1] > 0.15 else 'ok'
plt.title(f'Learning curves — {best_name}  ({diag})')
plt.xlabel('Training set size')
plt.ylabel('RMSE')
plt.legend()
plt.tight_layout(); plt.show()

print(f'train rmse : {tr_rmse.mean(axis=1)[-1]:.4f}')
print(f'cv rmse    : {val_rmse.mean(axis=1)[-1]:.4f}')
print(f'gap        : {gap:.4f}')


In [ ]:
train_pred_log = best_model.predict(X_train)
residuals      = y - train_pred_log

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(train_pred_log, residuals, alpha=0.3, color='steelblue', s=12)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuals vs fitted')
axes[0].set_xlabel('Fitted log(SalePrice)')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title(f'Residual distribution  (skew={residuals.skew():.3f})')
axes[1].set_xlabel('Residual')

(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
axes[2].scatter(osm, osr, alpha=0.4, s=10, color='mediumseagreen')
axes[2].plot(osm, slope*np.array(osm)+intercept, color='red', linewidth=1.5)
axes[2].set_title(f'QQ plot of residuals  R2={r**2:.4f}')
axes[2].set_xlabel('Theoretical quantiles')
axes[2].set_ylabel('Sample quantiles')

plt.suptitle(f'Residuals — {best_name}', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print(f'mean : {residuals.mean():.6f}')
print(f'std  : {residuals.std():.4f}')
print(f'skew : {residuals.skew():.4f}')


In [ ]:
test_preds = np.expm1(best_model.predict(X_test))

assert len(test_preds) == len(test_ids)
assert (test_preds > 0).all()

print(f'count      : {len(test_preds)}')
print(f'min        : ${test_preds.min():,.0f}')
print(f'max        : ${test_preds.max():,.0f}')
print(f'mean       : ${test_preds.mean():,.0f}')
print(f'train mean : ${np.expm1(y).mean():,.0f}')

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': test_preds})
submission.to_csv(os.path.join(CFG['output_dir'], 'submission.csv'), index=False)
submission.to_csv(os.path.join(CFG['output_dir'], f'submission_{RUN_TIMESTAMP}.csv'), index=False)
print(f'saved submission.csv  ({best_name})')
submission.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(train_clean['SalePrice'], bins=60, color='steelblue',
             edgecolor='white', alpha=0.7, label='train actual', density=True)
axes[0].hist(test_preds, bins=60, color='darkorange',
             edgecolor='white', alpha=0.6, label='test predicted', density=True)
axes[0].legend()
axes[0].set_title(f'Price distribution — {best_name}')
axes[0].set_xlabel('Sale price ($)')
axes[0].set_ylabel('Density')

axes[1].axis('off')
summary = [
    f'run        : {RUN_TIMESTAMP}',
    f'best model : {best_name}',
    f'cv rmse    : {results[best_name]:.4f}',
    f'train rows : {len(X_train)}',
    f'test rows  : {len(X_test)}',
    f'features   : {X_train.shape[1]}',
]
axes[1].text(0.1, 0.5, '\n'.join(summary), transform=axes[1].transAxes,
             fontsize=13, verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout(); plt.show()

joblib.dump(best_model, os.path.join(CFG['output_dir'], 'best_model.pkl'))
joblib.dump(best_model, os.path.join(CFG['output_dir'], f'best_model_{RUN_TIMESTAMP}.pkl'))
joblib.dump(fitted_models, os.path.join(CFG['output_dir'], 'all_models.pkl'))
print(f'saved best_model.pkl  all_models.pkl')


In [ ]:
estimator = best_model[-1] if isinstance(best_model, Pipeline) else best_model

if hasattr(estimator, 'feature_importances_'):
    imp   = pd.Series(estimator.feature_importances_, index=X_train.columns)
    top20 = imp.sort_values(ascending=False).head(20)
    kind  = 'importance'
elif hasattr(estimator, 'coef_'):
    top20 = pd.Series(np.abs(estimator.coef_), index=X_train.columns).sort_values(ascending=False).head(20)
    kind  = 'coefficient magnitude'
else:
    top20 = None
    kind  = None
    print('feature importance not available for this model')

if top20 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    top20.sort_values().plot(kind='barh', color='darkorange', edgecolor='white', ax=axes[0])
    axes[0].set_title(f'Top 20 features — {kind}  [{best_name}]')
    axes[0].set_xlabel(kind)

    cum_imp = top20.sort_values(ascending=False).cumsum() / top20.sum() * 100
    axes[1].plot(range(1, len(cum_imp)+1), cum_imp.values, 'o-', color='steelblue')
    axes[1].axhline(80, color='red', linestyle='--', label='80%')
    axes[1].set_title('Cumulative feature importance')
    axes[1].set_xlabel('Number of features')
    axes[1].set_ylabel('Cumulative importance (%)')
    axes[1].legend()

    plt.tight_layout(); plt.show()

    for rank, (feat, val) in enumerate(top20.sort_values(ascending=False).head(10).items(), 1):
        print(f'{rank:2}. {feat:<40} {val:.5f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

tc = train_clean.copy()
tc['IsRemodelled'] = (tc['YearBuilt'] != tc['YearRemodAdd']).astype(int)
sns.boxplot(x='IsRemodelled', y='SalePrice', data=tc,
            palette=['#4A90D9','#E86F68'], ax=axes[0])
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['not remodelled', 'remodelled'])
med0 = tc[tc['IsRemodelled']==0]['SalePrice'].median()
med1 = tc[tc['IsRemodelled']==1]['SalePrice'].median()
axes[0].set_title(f'Renovation lift: +${med1-med0:,.0f}  (+{(med1/med0-1)*100:.1f}%)')

cond = (tc.groupby('SaleCondition')['SalePrice'].median().sort_values(ascending=False))
cond.plot(kind='bar', color='mediumslateblue', edgecolor='white', ax=axes[1])
axes[1].set_title('Median price by sale condition')
axes[1].tick_params(axis='x', rotation=20)

tc['QualTier'] = pd.cut(tc['OverallQual'], bins=[0,3,6,8,10],
                         labels=['1-3','4-6','7-8','9-10'])
sns.boxplot(x='QualTier', y='SalePrice', data=tc, palette='RdYlGn', ax=axes[2])
axes[2].set_title('Price by quality tier')

for ax in axes: ax.set_ylabel('Sale price ($)')
plt.tight_layout(); plt.show()
